# Find the customers who never ordered

An `INNER JOIN` drops the rows you are looking for before `WHERE` can see them.

In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER);
INSERT INTO customers VALUES (1,'Anna'),(2,'Ben'),(3,'Cem'),(4,'Dana');
INSERT INTO orders VALUES (10,1),(11,1),(12,3);
""")

print("INNER:", db.execute("""
    SELECT c.name FROM customers c JOIN orders o ON o.customer_id = c.id WHERE o.id IS NULL
""").fetchall())

INNER: []


In [2]:
print("LEFT: ", db.execute("""
    SELECT c.name FROM customers c LEFT JOIN orders o ON o.customer_id = c.id WHERE o.id IS NULL
""").fetchall())

LEFT:  [('Ben',), ('Dana',)]


## What the two joins actually produce

In [3]:
for kind in ["JOIN", "LEFT JOIN"]:
    rows = db.execute(f"SELECT c.name, o.id FROM customers c {kind} orders o ON o.customer_id = c.id").fetchall()
    print(f"{kind:9}", rows)

JOIN      [('Anna', 10), ('Anna', 11), ('Cem', 12)]
LEFT JOIN [('Anna', 10), ('Anna', 11), ('Ben', None), ('Cem', 12), ('Dana', None)]


`NULL` means *absent*, and it matches nothing, not even itself.

In [4]:
print(db.execute("SELECT NULL = NULL, NULL IS NULL").fetchone())
print("COUNT(*)   per customer:", db.execute("SELECT c.name, COUNT(*)   FROM customers c LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.name").fetchall())
print("COUNT(o.id) per customer:", db.execute("SELECT c.name, COUNT(o.id) FROM customers c LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.name").fetchall())

(None, 1)
COUNT(*)   per customer: [('Anna', 2), ('Ben', 1), ('Cem', 1), ('Dana', 1)]
COUNT(o.id) per customer: [('Anna', 2), ('Ben', 0), ('Cem', 1), ('Dana', 0)]


`COUNT(*)` counts the NULL row as one order and `COUNT(o.id)` does not, which
is the difference between "Ben has 1 order" and the truth.

**What we used:** INNER versus LEFT joins, NULL, and the order in which a query
is evaluated (session 11, sections 11 and 12).